# Sesi 2 — Knowledge Agent: CRUD REST API

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen + Gemini**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya (bisa disembunyikan saat sesi diberikan ke peserta).

## 0. Setup Environment

In [ ]:
!pip -q install duckdb sentence-transformers fastapi httpx nest_asyncio uvicorn

## 1. Rekonstruksi Koneksi & Model dari Sesi 1
Jika `knowledge.duckdb` dari Sesi 1 belum ada, sel di bawah akan membuat ulang skema dan mengisi beberapa contoh dokumen.

In [ ]:
import duckdb, uuid, os
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
con = duckdb.connect("knowledge.duckdb")
con.execute("INSTALL vss; LOAD vss;")
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id VARCHAR PRIMARY KEY, source VARCHAR, content TEXT,
    embedding FLOAT[384], created_at TIMESTAMP DEFAULT current_timestamp
);
""")

if con.execute("SELECT COUNT(*) FROM documents").fetchone()[0] == 0:
    seed = [
        ("faq_produk.txt", "Garansi produk berlaku 1 tahun sejak tanggal pembelian."),
        ("sop_retur.txt", "Retur barang maksimal 7 hari setelah barang diterima."),
    ]
    for source, text in seed:
        emb = embed_model.encode(text).tolist()
        con.execute("INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
                    [str(uuid.uuid4()), source, text, emb])
print(con.execute("SELECT COUNT(*) FROM documents").fetchone())

## 2. Bangun FastAPI App — CRUD `documents`

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title="Knowledge Agent API")

class DocIn(BaseModel):
    source: str
    content: str

class DocOut(DocIn):
    id: str

@app.post("/documents", response_model=DocOut)
def create_document(doc: DocIn):
    doc_id = str(uuid.uuid4())
    emb = embed_model.encode(doc.content).tolist()
    con.execute("INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
                [doc_id, doc.source, doc.content, emb])
    return {**doc.dict(), "id": doc_id}

@app.get("/documents", response_model=List[DocOut])
def list_documents(limit: int = 20, offset: int = 0):
    rows = con.execute("SELECT id, source, content FROM documents LIMIT ? OFFSET ?",
                       [limit, offset]).fetchall()
    return [{"id": r[0], "source": r[1], "content": r[2]} for r in rows]

@app.get("/documents/search")
def search_documents(q: str, k: int = 5):
    q_emb = embed_model.encode(q).tolist()
    rows = con.execute("""
        SELECT id, source, content, array_distance(embedding, ?::FLOAT[384]) AS dist
        FROM documents ORDER BY dist ASC LIMIT ?
    """, [q_emb, k]).fetchall()
    return [{"id": r[0], "source": r[1], "content": r[2], "score": r[3]} for r in rows]

@app.get("/documents/{doc_id}", response_model=DocOut)
def get_document(doc_id: str):
    row = con.execute("SELECT id, source, content FROM documents WHERE id=?", [doc_id]).fetchone()
    if not row:
        raise HTTPException(404, "Document not found")
    return {"id": row[0], "source": row[1], "content": row[2]}

@app.put("/documents/{doc_id}", response_model=DocOut)
def update_document(doc_id: str, doc: DocIn):
    emb = embed_model.encode(doc.content).tolist()
    con.execute("UPDATE documents SET content=?, source=?, embedding=? WHERE id=?",
                [doc.content, doc.source, emb, doc_id])
    return {**doc.dict(), "id": doc_id}

@app.delete("/documents/{doc_id}")
def delete_document(doc_id: str):
    con.execute("DELETE FROM documents WHERE id=?", [doc_id])
    return {"status": "deleted", "id": doc_id}

### 🔨 TODO 1
Tambahkan endpoint `POST /documents/bulk` yang menerima `List[DocIn]` dan meng-insert semuanya sekaligus, mengembalikan jumlah dokumen yang berhasil disimpan.

In [ ]:
# TODO: implementasikan endpoint bulk insert
@app.post("/documents/bulk")
def bulk_create(___):
    pass

**Solusi contoh:**
```python
@app.post("/documents/bulk")
def bulk_create(docs: List[DocIn]):
    for doc in docs:
        create_document(doc)
    return {"inserted": len(docs)}
```

## 3. Uji API Tanpa Server Eksternal (TestClient)
`TestClient` menjalankan FastAPI langsung di memori — cocok untuk Colab tanpa perlu ngrok.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

resp = client.post("/documents", json={"source": "manual.txt", "content": "Jam operasional toko 09.00-21.00 setiap hari."})
print(resp.status_code, resp.json())

In [ ]:
resp = client.get("/documents/search", params={"q": "jam buka toko", "k": 3})
resp.json()

## 4. (Opsional) Jalankan sebagai Server Publik via ngrok
Berguna kalau ingin API ini dipanggil dari luar Colab (misalnya di Sesi 3–4).

In [ ]:
# !pip -q install pyngrok
# import nest_asyncio, uvicorn
# from pyngrok import ngrok
# nest_asyncio.apply()
# ngrok.set_auth_token("<NGROK_AUTH_TOKEN>")
# public_url = ngrok.connect(8001)
# print("Public URL:", public_url)
# uvicorn.run(app, port=8001)

## Ringkasan Sesi 2
API CRUD + search untuk Knowledge Agent sudah siap (`app` FastAPI di memori Colab / opsional publik via ngrok). Sesi 3 akan memakai `client` (atau `public_url`) ini sebagai *tool* dalam ReAct loop.